In [ ]:
# ============================================================
# CELL 1 — Install dependencies
# ============================================================
! pip install av

In [ ]:
# ============================================================
# CELL 2 — Imports
# ============================================================
import wandb
import av
import numpy as np
import json
import os,shutil
import torch
from tqdm import tqdm
import matplotlib.pyplot as plt
import random 
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
from transformers import get_cosine_schedule_with_warmup
# from peft import get_peft_model, LoraConfig, TaskType
from transformers import AutoImageProcessor, AutoModelForSeq2SeqLM
from transformers import TimesformerModel, NllbTokenizerFast
from transformers.modeling_outputs import BaseModelOutput
from transformers import TrainingArguments, Trainer, TrainerCallback


In [ ]:
# # # ============================================================
# # # CELL 3 — WandB API Key (uncomment to use)
# # # ============================================================
os.environ["WANDB_API_KEY"] = "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"

In [ ]:

# ============================================================
# CELL 4 — Paths
# ============================================================
dataset_folder    = "/workspace/VATEX_ne"
video_folder_path = os.path.join(dataset_folder, "videos_train_val")
train_json_path   = os.path.join(dataset_folder, "nepali_captions/vatex_train_ne.json")
val_json_path     = os.path.join(dataset_folder, "nepali_captions/vatex_val_ne.json")
test_json_path    = os.path.join(dataset_folder, "nepali_captions/vatex_test_ne.json")
cache_dir         = "/workspace/video_tensor_cache"

In [ ]:
# ============================================================
# CELL 5 — Video Reading (16 frames, up from 8)
# ============================================================
def read_video_pyav(video_path, num_frames=8):   # FIX: 8 → 16
    container = av.open(video_path)
    frames = []

    for frame in container.decode(video=0):
        frames.append(frame.to_rgb().to_ndarray())

    container.close()

    idx    = np.linspace(0, max(len(frames) - 1, 0), num_frames).astype(np.int64)
    frames = [frames[i] for i in idx]

    return frames  # (T, H, W, C)


In [ ]:

# ============================================================
# CELL 6 — Tokenizer + Image Processor
# FIX: replaced MBart50TokenizerFast with NllbTokenizerFast
# ============================================================
tokenizer = NllbTokenizerFast.from_pretrained(
    "facebook/nllb-200-distilled-600M",
    src_lang="npi_Deva",
    tgt_lang="npi_Deva"
)

image_processor = AutoImageProcessor.from_pretrained(
    "MCG-NJU/videomae-base",
    use_fast=True
)


In [ ]:

# ============================================================
# CELL 7 — Video Processing
# ============================================================
def process_video(video_path, image_processor):
    frames = read_video_pyav(video_path)
    inputs = image_processor(list(frames), return_tensors="pt")
    return inputs["pixel_values"].half()  # fp16, shape: (1, T, C, H, W)


In [ ]:

# # ============================================================
# # CELL 8 — Make Caption Pairs
# # ============================================================
# def make_video_caption_pairs(json_path):
#     pairs = []
#     with open(json_path) as f:
#         data = json.load(f)
#     for item in data:
#         video_id = item["video_id"]
#         captions = item["nepali_captions"]
#         for caption in captions:
#             pairs.append({
#                 "video_id": video_id,
#                 "caption":  caption
#             })
#     return pairs


# ============================================================
# CELL 8 — Make Caption Pairs (GROUPED)
# ============================================================
def make_video_caption_pairs(json_path):
    with open(json_path) as f:
        data = json.load(f)
    
    pairs = []
    for item in data:
        # We store all captions in a list for the sampler to use
        pairs.append({
            "video_id": item["video_id"],
            # "captions": item["nepali_captions"] 
             "captions": item["caption"] 
        })
    return pairs

In [ ]:

# ============================================================
# CELL 9 — VideoOnly Dataset (for caching)
# ============================================================
class VideoOnlyDataset(Dataset):
    def __init__(self, pairs, video_dir, cache_dir, processor):
        seen       = set()
        self.items = []
        for p in pairs:
            if p["video_id"] not in seen:
                if not os.path.exists(os.path.join(cache_dir, f"{p['video_id']}.pt")):
                    seen.add(p["video_id"])
                    self.items.append(p["video_id"])
        self.video_dir = video_dir
        self.cache_dir = cache_dir
        self.processor = processor

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        video_id   = self.items[idx]
        video_path = os.path.join(self.video_dir, f"{video_id}.mp4")
        tensor     = process_video(video_path, self.processor)  # (1, T, C, H, W) fp16
        tensor     = tensor.squeeze(0)                          # (T, C, H, W) fp16
        return video_id, tensor


In [ ]:


# ============================================================
# CELL 10 — Cache Builder
# ============================================================
def cache_collate_fn(batch):
    video_ids = [item[0] for item in batch]
    tensors   = torch.stack([item[1] for item in batch])
    return video_ids, tensors


def build_video_cache(pairs, video_dir, cache_dir, processor, num_workers=2, batch_size=4):
    os.makedirs(cache_dir, exist_ok=True)
    print(f"Building cache in: {cache_dir}")

    dataset = VideoOnlyDataset(pairs, video_dir, cache_dir, processor)

    if len(dataset) == 0:
        print("All videos already cached ✅")
        return

    loader = DataLoader(
        dataset,
        batch_size  = batch_size,
        num_workers = num_workers,
        shuffle     = False,
        collate_fn  = cache_collate_fn
    )

    print(f"Caching {len(dataset)} videos...")

    for video_ids, tensors in tqdm(loader, total=len(loader)):
        for video_id, tensor in zip(video_ids, tensors):
            cache_path = os.path.join(cache_dir, f"{video_id}.pt")
            torch.save(tensor.clone(), cache_path, _use_new_zipfile_serialization=False)

    print("Done ✅")

In [ ]:


# ============================================================
# CELL 11 — Build pairs + cache
# ============================================================
train_pairs = make_video_caption_pairs(train_json_path)
val_pairs   = make_video_caption_pairs(val_json_path)

build_video_cache(
    pairs       = train_pairs,
    video_dir   = video_folder_path,
    cache_dir   = cache_dir,
    processor   = image_processor,
    num_workers = 8,
    batch_size  = 32
)
build_video_cache(
    pairs       = val_pairs,
    video_dir   = video_folder_path,
    cache_dir   = cache_dir,
    processor   = image_processor,
    num_workers = 8,
    batch_size  = 32
)

In [ ]:

# ============================================================
# CELL 12 — Verify cache
# ============================================================
files = os.listdir(cache_dir)
f     = torch.load(os.path.join(cache_dir, files[0]), weights_only=False)
print("shape:", f.shape)   # should be [16, 3, 224, 224]
print("dtype:", f.dtype)   # should be torch.float16
print("size :", os.path.getsize(os.path.join(cache_dir, files[0])) / 1e6, "MB")

In [ ]:
import psutil

def get_ram_usage():
    ram = psutil.virtual_memory()
    print(f"Used RAM : {ram.used/1e9:.1f} GB")
    print(f"Free RAM : {ram.available/1e9:.1f} GB")

print("=== BEFORE ===")
get_ram_usage()

video_cache = {}
for filename in tqdm(os.listdir(cache_dir)):
    if filename.endswith(".pt"):
        video_id = filename.replace(".pt", "")
        video_cache[video_id] = torch.load(
            os.path.join(cache_dir, filename),
            weights_only=False,
            map_location="cpu"
        )

print("=== AFTER ===")
get_ram_usage()
print(f"Videos loaded: {len(video_cache)}")

In [ ]:
# # ============================================================
# # CELL 13 — Dataset with Random Sampling
# # ============================================================
# import random

# class VideoCaptionDataset(Dataset):
#     def __init__(self, pairs, cache_dir, tokenizer, max_length=64, is_training=True):
#         self.pairs = pairs
#         self.cache_dir = cache_dir
#         self.tokenizer = tokenizer
#         self.max_length = max_length
#         self.is_training = is_training

#     def __len__(self):
#         return len(self.pairs)

#     def __getitem__(self, idx):
#         item = self.pairs[idx]
#         video_id = item["video_id"]
        
#         # Training: Pick a random one from the 10-40 available
#         # Eval: Stick to the first one for consistency
#         if self.is_training and len(item["captions"]) > 0:
#             caption = random.choice(item["captions"])
#         else:
#             caption = item["captions"][0]

#         # Load cached tensor
#         cache_path = os.path.join(self.cache_dir, f"{video_id}.pt")
#         video_tensor = torch.load(cache_path, map_location="cpu", weights_only=False)
#         video_tensor = video_tensor.float()

#         text_inputs = self.tokenizer(
#             caption,
#             padding="max_length",
#             truncation=True,
#             max_length=self.max_length,
#             return_tensors="pt"
#         )
        
#         input_ids = text_inputs["input_ids"].squeeze(0)
#         labels = input_ids.clone()
#         labels[labels == self.tokenizer.pad_token_id] = -100

#         return {
#             "pixel_values": video_tensor,
#             "input_ids": input_ids,
#             "labels": labels
#         }

class VideoCaptionDataset(Dataset):
    def __init__(self, pairs, video_cache, tokenizer, max_length=64, is_training=True):
        self.pairs       = pairs
        self.video_cache   = video_cache
        self.tokenizer   = tokenizer
        self.max_length  = max_length
        self.is_training = is_training
        self.epoch       = 0

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        item       = self.pairs[idx]
        video_id   = item["video_id"]
        captions   = item["captions"]

        if self.is_training:
            caption = random.choice(captions)          # full diversity
        else:
            caption = captions[self.epoch % len(captions)]  # rotation

        video_tensor = video_cache[video_id].float()

        text_inputs = self.tokenizer(
            caption,
            padding      = "max_length",
            truncation   = True,
            max_length   = self.max_length,
            return_tensors = "pt"
        )
        input_ids = text_inputs["input_ids"].squeeze(0)
        labels    = input_ids.clone()
        labels[labels == self.tokenizer.pad_token_id] = -100

        return {
            "pixel_values": video_tensor,
            "input_ids":    input_ids,
            "labels":       labels,
        }

In [ ]:
# ============================================================
# CELL 14 — Instantiate datasets
# ============================================================
train_dataset = VideoCaptionDataset(train_pairs, video_cache, tokenizer)
val_dataset   = VideoCaptionDataset(val_pairs,video_cache, tokenizer)

print(f"Train size: {len(train_dataset)}")
print(f"Val size:   {len(val_dataset)}")


In [ ]:


# ============================================================
# CELL 15 — Quick sanity checks
# ============================================================
sample = train_dataset[0]
print("pixel_values shape:", sample["pixel_values"].shape)  # [16, 3, 224, 224]
print("input_ids shape   :", sample["input_ids"].shape)     # [64]
print("labels shape      :", sample["labels"].shape)        # [64]
print("Decoded caption   :", tokenizer.decode(
    sample["input_ids"][sample["input_ids"] != tokenizer.pad_token_id],
    skip_special_tokens=True
))


In [ ]:
# Ensure tensor is on CPU and float32
frames = sample["pixel_values"].detach().cpu().float()  # [T, C, H, W]

# Denormalize frames (inverse of typical ImageNet normalization)
mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
frames = frames * std + mean

# Clamp values to [0,1] for safe visualization
frames = frames.clamp(0, 1)

# Convert to numpy and rearrange dimensions for matplotlib
frames = frames.numpy()  # [T, C, H, W]
frames = np.transpose(frames, (0, 2, 3, 1))  # [T, H, W, C]

# Create subplots
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flat  # Flatten axes for iteration

for idx, ax in enumerate(axes):
    if idx < len(frames):
        ax.imshow(frames[idx])
        ax.set_title(f'Frame {idx+1}')
        ax.axis('off')
    else:
        ax.axis('off')

plt.suptitle('Video Frames (8 frames)', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
text = tokenizer.decode(sample["input_ids"])
print("Decoded Text:", text)

In [ ]:

# ============================================================
# CELL 16 — Q-Former Block WITH dropout
# FIX: added dropout throughout
# ============================================================
class QFormerBlock(nn.Module):
    def __init__(self, hidden_dim=1024, num_heads=8, mlp_ratio=4.0, dropout=0.1):
        super().__init__()

        self.cross_attn = nn.MultiheadAttention(
            embed_dim   = hidden_dim,
            num_heads   = num_heads,
            batch_first = True,
            dropout     = dropout,        # FIX: added dropout
        )
        self.norm1 = nn.LayerNorm(hidden_dim)

        self.self_attn = nn.MultiheadAttention(
            embed_dim   = hidden_dim,
            num_heads   = num_heads,
            batch_first = True,
            dropout     = dropout,        # FIX: added dropout
        )
        self.norm2 = nn.LayerNorm(hidden_dim)

        self.mlp = nn.Sequential(
            nn.Linear(hidden_dim, int(hidden_dim * mlp_ratio)),
            nn.GELU(),
            nn.Dropout(dropout),          # FIX: added dropout in MLP
            nn.Linear(int(hidden_dim * mlp_ratio), hidden_dim),
        )
        self.norm3   = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, queries, video_features):
        cross_out, _ = self.cross_attn(
            query=queries,
            key  =video_features,
            value=video_features,
        )
        queries = self.norm1(queries + self.dropout(cross_out))   # FIX: dropout

        self_out, _ = self.self_attn(
            query=queries,
            key  =queries,
            value=queries,
        )
        queries = self.norm2(queries + self.dropout(self_out))    # FIX: dropout

        mlp_out = self.mlp(queries)
        queries = self.norm3(queries + self.dropout(mlp_out))     # FIX: dropout

        return queries


# ============================================================
# CELL 17 — Video Captioning Model
# FIX: replaced mBART with NLLB-600M, partial TimeSformer unfreeze,
#      frozen decoder embeddings
# ============================================================
class VideoCaptioningModel(nn.Module):
    def __init__(
        self,
        timesformer_model_name = "facebook/timesformer-base-finetuned-k600",
        nllb_model_name        = "facebook/nllb-200-distilled-600M",  # FIX: was mBART
        num_query_tokens       = 50,
        qformer_layers         = 4,
        dropout                = 0.1,
        freeze_timesformer     = True,
        unfreeze_last_n_blocks = 3,           # FIX: partial unfreeze
        freeze_nllb_encoder    = True,
    ):
        super().__init__()

        # -------- Encoder: TimeSformer --------
        self.encoder = TimesformerModel.from_pretrained(timesformer_model_name)

        # -------- Decoder: NLLB-600M (FIX: was mBART) --------
        self.decoder = AutoModelForSeq2SeqLM.from_pretrained(nllb_model_name)

        ts_hidden   = self.encoder.config.hidden_size   # 768
        nllb_hidden = self.decoder.config.d_model       # 1024

        # -------- Projection (768 → 1024) --------
        self.video_proj = nn.Linear(ts_hidden, nllb_hidden)

        # -------- Q-Former --------
        self.query_tokens = nn.Parameter(
            torch.randn(1, num_query_tokens, nllb_hidden)
        )
        self.qformer_blocks = nn.ModuleList([
            QFormerBlock(hidden_dim=nllb_hidden, dropout=dropout)
            for _ in range(qformer_layers)
        ])

        # -------- Freeze TimeSformer --------
        if freeze_timesformer:
            for p in self.encoder.parameters():
                p.requires_grad = False
        
            # Unfreeze last 3 blocks
            for block in self.encoder.encoder.layer[-unfreeze_last_n_blocks:]:
                for p in block.parameters():
                    p.requires_grad = True
        
            print(f"TimeSformer: frozen except last {unfreeze_last_n_blocks} blocks ✅")
        
        # -------- Freeze NLLB --------
        if freeze_nllb_encoder:
            # Freeze entire encoder (not used anyway)
            for p in self.decoder.model.encoder.parameters():
                p.requires_grad = False
        
            # Train decoder layers
            for p in self.decoder.model.decoder.parameters():
                p.requires_grad = True
        
            # # Freeze embeddings (already knows Nepali perfectly)
            # for p in self.decoder.model.shared.parameters():
            #     p.requires_grad = False
        
            # # Freeze lm_head (same matrix as shared embeddings)
            # for p in self.decoder.lm_head.parameters():
            #     p.requires_grad = False
                
            #     # Freeze lm_head (same matrix as shared embeddings)
            # for p in self.decoder.model.decoder.embed_tokens.parameters():
            #     p.requires_grad = False
                
        
            # print("NLLB: encoder frozen, decoder trainable, embeddings frozen ✅")
        
        # -------- Verify --------
        # total     = sum(p.numel() for p in self.parameters())
        # trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        # print(f"Total params:     {total/1e6:.1f}M")
        # print(f"Trainable params: {trainable/1e6:.1f}M")
        # print(f"Frozen params:    {(total-trainable)/1e6:.1f}M")

    # --------------------------------------------------
    # Forward (Training)
    # --------------------------------------------------
    def forward(self, pixel_values, input_ids, labels):

        # 1. TimeSformer Encoding
        encoder_outputs = self.encoder(
            pixel_values=pixel_values,
            return_dict =True
        )
        video_features = encoder_outputs.last_hidden_state  # (B, N, 768)

        # 2. Project to NLLB hidden size
        video_features = self.video_proj(video_features)    # (B, N, 1024)

        B = video_features.size(0)

        # 3. Expand Query Tokens
        queries = self.query_tokens.expand(B, -1, -1)       # (B, 32, 1024)

        # 4. Q-Former Processing
        for block in self.qformer_blocks:
            queries = block(queries, video_features)         # (B, 32, 1024)

        # 5. Wrap as Encoder Memory for NLLB decoder
        encoder_out = BaseModelOutput(last_hidden_state=queries)
        encoder_attention_mask = torch.ones(
            queries.size()[:-1],
            dtype =torch.long,
            device=queries.device
        )

        # 6. NLLB Decoder forward
        outputs = self.decoder(
            encoder_outputs  =encoder_out,
            attention_mask   =encoder_attention_mask,
            input_ids        =input_ids,
            labels           =labels,
            return_dict      =True
        )

        return outputs

    # --------------------------------------------------
    # Generate (Inference)
    # --------------------------------------------------
    @torch.no_grad()
    def generate(
        self,
        pixel_values,
        max_length           = 50,            # FIX: was 32
        num_beams            = 4,
        forced_bos_token_id  = None,
        **generate_kwargs
    ):
        encoder_outputs = self.encoder(
            pixel_values=pixel_values,
            return_dict =True
        )
        video_features = self.video_proj(encoder_outputs.last_hidden_state)

        B       = video_features.size(0)
        queries = self.query_tokens.expand(B, -1, -1)

        for block in self.qformer_blocks:
            queries = block(queries, video_features)

        encoder_out = BaseModelOutput(last_hidden_state=queries)
        encoder_attention_mask = torch.ones(
            queries.size()[:-1],
            dtype =torch.long,
            device=queries.device
        )

        generated_ids = self.decoder.generate(
            encoder_outputs  =encoder_out,
            attention_mask   =encoder_attention_mask,
            max_length       =max_length,
            num_beams        =num_beams,
            forced_bos_token_id=forced_bos_token_id,
            no_repeat_ngram_size=3,
            early_stopping   =True,
            **generate_kwargs
        )

        return generated_ids





In [ ]:
# ============================================================
# CELL 18 — Instantiate model
# ============================================================
torch.cuda.empty_cache()

model = VideoCaptioningModel(
    timesformer_model_name = "facebook/timesformer-base-finetuned-k600",
    nllb_model_name        = "facebook/nllb-200-distilled-600M",
    num_query_tokens       = 50,
    qformer_layers         = 4,
    dropout                = 0.1,
    freeze_timesformer     = True,
    unfreeze_last_n_blocks = 3,
    freeze_nllb_encoder    = True,
)

In [ ]:



# lora_config = LoraConfig(
#     task_type      = TaskType.SEQ_2_SEQ_LM,
#     r              = 16,
#     lora_alpha     = 32,
#     lora_dropout   = 0.1,
#     target_modules = ["q_proj", "v_proj", "k_proj"],  # self + cross attn
#     bias           = "none",
# )
# model.decoder = get_peft_model(model.decoder, lora_config)

# for param in model.decoder.base_model.model.model.encoder.parameters():
#     param.requires_grad = False

In [ ]:
# print(f"{'Layer Name':<60} | {'Trainable?':<12} | {'Param Count'}")
# print("-" * 85)

# for name, param in model.named_parameters():
#     # Filter for decoder to keep the list manageable
#     if "decoder" in name:
#         trainable = "✅ YES" if param.requires_grad else "❌ NO"
#         # Convert to Millions for readability
#         num_params = f"{param.numel()/1e6:.4f}M"
#         print(f"{name[:60]:<60} | {trainable:<12} | {num_params}")

# # Quick summary of the Cross-Attention specifically
# cross_attn_trainable = any("encoder_attn" in n and p.requires_grad for n, p in model.named_parameters())
# print("\n" + "="*30)
# print(f"CROSS-ATTENTION TRAINABLE: {cross_attn_trainable}")
# print("="*30)

In [ ]:
def print_param_counts(timesformer_list, adapter_list):
    ts_params = sum(p.numel() for p in timesformer_list)
    ad_params = sum(p.numel() for p in adapter_list)
    
    print(f"--- Trainable Parameter Breakdown ---")
    print(f"TimeSformer:    {ts_params/1e6:.2f}M parameters ({len(timesformer_list)} tensors)")
    print(f"Adapter/Decoder: {ad_params/1e6:.2f}M parameters ({len(adapter_list)} tensors)")
    print(f"Total Trainable: {(ts_params + ad_params)/1e6:.2f}M")

In [ ]:
for name, param in model.named_parameters():
    if any(x in name for x in ["shared", "lm_head", "embed_tokens"]):
        print(f"{name}: requires_grad={param.requires_grad}")

In [ ]:
def print_model_memory(model):
    total_params = 0
    trainable_params = 0
    
    for name, param in model.named_parameters():
        total_params += param.numel()
        if param.requires_grad:
            trainable_params += param.numel()
    
    print(f"Total params:     {total_params/1e6:.1f}M")
    print(f"Trainable params: {trainable_params/1e6:.1f}M")
    print(f"Frozen params:    {(total_params-trainable_params)/1e6:.1f}M")
    
    # Memory estimates
    total_mb     = total_params * 2 / 1e6        # fp16 = 2 bytes
    trainable_mb = trainable_params * 2 / 1e6
    optimizer_mb = trainable_params * 8 / 1e6    # AdamW = 4x fp32
    
    print(f"\nWeights (fp16):         {total_mb:.0f} MB")
    print(f"Trainable weights:      {trainable_mb:.0f} MB")
    print(f"Optimizer states (fp32):{optimizer_mb:.0f} MB")
    print(f"Estimated total:        {(total_mb + optimizer_mb):.0f} MB")

print_model_memory(model)

In [ ]:
# See exactly what is trainable
for name, param in model.named_parameters():
    if param.requires_grad:
        print(f"{name}: {param.numel()/1e6:.1f}M")

In [ ]:
# ============================================================
# CELL 19 — Data Collator
# ============================================================
def data_collator(batch):
    return {
        "pixel_values": torch.stack([item["pixel_values"] for item in batch]),
        "input_ids":    torch.stack([item["input_ids"]    for item in batch]),
        "labels":       torch.stack([item["labels"]       for item in batch]),
    }


In [ ]:
# from huggingface_hub import snapshot_download

# # Download only the checkpoint-1000 folder
# local_ckpt_path = snapshot_download(
#     repo_id    = "sulavsss7/run2-updated-lora-nllb",
#     allow_patterns = "checkpoint-2025/*",
#     local_dir  = "/tmp/checkpoints",
#     token      = secret_value_0
# )

# print(f"✅ Checkpoint downloaded to: /tmp/checkpoints/checkpoint-2025")

In [ ]:


# # ============================================================
# # CELL 20 — WandB Init
# # ============================================================
# wandb.init(
#     project="huggingface",
#     id     ="fdhu7dmy",
#     resume ="allow",
#     settings=wandb.Settings(init_timeout=300)
# )

In [ ]:
class EpochUpdateCallback(TrainerCallback):
    def __init__(self, train_dataset, val_dataset):
        self.train_dataset = train_dataset
        self.val_dataset   = val_dataset

    def on_epoch_begin(self, args, state, control, **kwargs):
        epoch = int(state.epoch)
        self.train_dataset.epoch = epoch
        self.val_dataset.epoch   = epoch
        print(f"Epoch {epoch}: val caption index → {epoch % 40}")

In [ ]:
# # ============================================================
# # CELL 22 — Training Arguments (Simplified for 30 Epochs)
# # ============================================================

# training_args = TrainingArguments(
#     output_dir                  = "/tmp/checkpoints",
#     push_to_hub                 = True,
#     hub_model_id                = "sulavsss7/run2-updated-lora-nllb",
#     hub_strategy                = "all_checkpoints",
#     hub_token                   = secret_value_0,

#     per_device_train_batch_size = 4,
#     gradient_accumulation_steps = 4,
#     per_device_eval_batch_size  = 4,

#     num_train_epochs            = 30,

#     fp16                        = True,
#     dataloader_num_workers      = 4,
#     dataloader_pin_memory       = True,

#     logging_strategy            = "epoch",
#     eval_strategy               = "epoch",
#     save_strategy               = "epoch",

#     save_total_limit            = 3,
#     load_best_model_at_end      = True,
#     metric_for_best_model       = "loss",
#     greater_is_better           = False,

#     report_to                   = "wandb",
#     run_name                    = "LORA-fixed-msvd-nllb-qt64-sampling",
# )

training_args = TrainingArguments(
    output_dir="/root/checkpoints",
    #Training
    num_train_epochs = 15,
    per_device_train_batch_size = 64,
    per_device_eval_batch_size = 64,
    gradient_accumulation_steps=2,

    #Optimization
    learning_rate = 1e-5,
    weight_decay = 0.001,
    fp16 = True,

    #Evaluation

    eval_strategy = "steps",
    eval_steps = 1000,
    save_strategy = "steps",
    save_steps = 1000,
    logging_strategy = "steps",
    logging_steps = 1000,

    load_best_model_at_end = True,
    metric_for_best_model = "eval_loss",
    greater_is_better = False,
    save_total_limit = 3,
    report_to = "wandb",
    run_name = "tf-qformer-nllb-vatex-capsamp",
)

In [ ]:
# # ============================================================
# # CELL 23 — Trainer
# # ============================================================


# trainer = Trainer(
#     model           = model,
#     args            = training_args,
#     train_dataset   = train_dataset,
#     eval_dataset    = val_dataset,
#     data_collator   = data_collator,
#     processing_class= tokenizer,
#     callbacks        = [
#         DynamicLogEvalCallback(),
#     ]
# )

# ============================================================
# CELL 23 — Trainer
# ============================================================
trainer = Trainer(
    model            = model,
    args             = training_args,
    train_dataset    = train_dataset,
    eval_dataset     = val_dataset,
    data_collator    = data_collator,
    processing_class = tokenizer,
    callbacks        = [EpochUpdateCallback(train_dataset, val_dataset)],
)

In [ ]:
trainer.train()  # use this for fresh training